<a href="https://colab.research.google.com/github/nifm093-dev/Fine_tuning_Module_level/blob/main/ApplyingRAGon_module_level_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Install (run once)
!pip install sentence-transformers scikit-learn -q

In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
import re

# Load your data (upload CSV first or paste snippet)
df = pd.read_csv("/content/uwaterloo_2000_classified.csv")  # replace with your file
df['nss_labels_clean'] = df['nss_labels'].str.lower().str.replace(r'labels:\s*', '', regex=True)

print(f"Loaded {len(df)} reviews")
print(df[['course_code', 'review_text', 'nss_labels_clean']].head())

def build_theme_retriever(df):
    """Simple in-memory retriever: filter by labels + embed"""
    model = SentenceTransformer('all-MiniLM-L6-v2')

    def retrieve(theme, top_k=10):
        # Filter docs with matching theme
        theme_lower = theme.lower()
        matching = df[df['nss_labels_clean'].str.contains(theme_lower, na=False)]

        if len(matching) == 0:
            return []

        # Embed texts
        texts = matching['review_text'].tolist()
        embeddings = model.encode(texts)

        return matching, embeddings, texts

    return model, retrieve


def select_inserts(model, query, matching_df, embeddings, texts, k=3, threshold=0.75):
    """Select 'close' inserts per supervisor feedback"""
    query_emb = model.encode(query)
    sims = cosine_similarity([query_emb], embeddings)[0]

    # Pairs: (sim_score, text, row)
    candidates = [(sims[i], texts[i], matching_df.iloc[i]) for i in range(len(texts))]

    # Filter threshold + top-k
    close = [(s, t, row) for s, t, row in candidates if s >= threshold]
    close.sort(reverse=True)

    selected = close[:k]
    return selected

# Example usage: Generate report for "Academic staff and support"
model, retriever = build_theme_retriever(df)

theme = "Academic staff and support"  # from your tables
query = f"negative comments on {theme.lower()}"

matching_df, embeddings, texts = retriever(theme)
inserts = select_inserts(model, query, matching_df, embeddings, texts)

print(f"\n📊 {len(matching_df)} reviews match '{theme}':")
print(f"Selected {len(inserts)} inserts (sim ≥0.75):")

for i, (sim, text, row) in enumerate(inserts):
    print(f"{i+1}. Sim={sim:.3f} | Course={row['course_code']} | '{text[:100]}...'")

# Generate report prompt (use your fine-tuned Llama or Grok)
report_prompt = f"""
Generate NSS-style report section for: {theme}
Stats: {len(matching_df)} mentions ({len(matching_df)/len(df)*100:.1f}%)

Use ONLY these evidence quotes:
{chr(10).join([f"- Sim={s:.2f}: {t}" for s,t,_ in inserts])}

Format like original: ranked insights, percentages, subtopics.
"""

print("\n🤖 Report Prompt:\n" + report_prompt)


Loaded 2000 reviews
  course_code                                        review_text  \
0      CS 115                    go to office hours and practice   
1      CS 115  One of my least favourite courses. Although th...   
2      CS 115  It starts with a very low pace but after midte...   
3      CS 115  Took this in 2018 with no programming experien...   
4      CS 115  I loved everything about cs 115. Great instruc...   

                                    nss_labels_clean  
0  learning opportunities, assessment and feedbac...  
1  learning community, learning resources, studen...  
2                                learning community,  
3  learning community, academic staff and support...  
4         covid-19 pandemic, learning opportunities,  


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


📊 659 reviews match 'Academic staff and support':
Selected 0 inserts (sim ≥0.75):

🤖 Report Prompt:

Generate NSS-style report section for: Academic staff and support
Stats: 659 mentions (33.0%)

Use ONLY these evidence quotes:


Format like original: ranked insights, percentages, subtopics.



In [4]:
# ========================================
# COMPLETE RAG FOR NSS REPORT GENERATION
# ========================================
# Upload your CSV, run → gets theme stats + selected inserts + report prompts
# Addresses supervisor: proper insert selection (sim threshold)

# Install (one-time)
!pip install sentence-transformers scikit-learn -q

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
import re

# 1. LOAD DATA
df = pd.read_csv("/content/uwaterloo_2000_classified.csv")  # UPLOAD YOUR 2000-ROW CSV HERE
df['nss_labels_clean'] = df['nss_labels'].astype(str).str.lower().str.replace(r'labels?:\s*', '', regex=True)

print(f"✅ Loaded {len(df)} reviews")
print("\nSample:")
print(df[['course_code', 'review_text', 'nss_labels_clean']].head(3))

# 2. BUILD RETRIEVER
model = SentenceTransformer('all-MiniLM-L6-v2')

def retrieve(theme, df=df):
    """Filter + embed reviews matching theme"""
    theme_lower = theme.lower()
    mask = df['nss_labels_clean'].str.contains(theme_lower, na=False)
    matching = df[mask].copy()

    if len(matching) == 0:
        print(f"⚠️ No reviews match '{theme}'")
        return pd.DataFrame(), np.array([]), []

    texts = matching['review_text'].fillna('').astype(str).tolist()
    embeddings = model.encode(texts)
    return matching, embeddings, texts

def select_inserts(query, matching_df, embeddings, texts, k=3, threshold=0.65):
    """Select 'close' inserts (supervisor method)"""
    query_emb = model.encode(query)
    sims = cosine_similarity([query_emb], embeddings)[0]

    candidates = [(sims[i], texts[i], matching_df.iloc[i]) for i in range(len(texts))]
    close = [(s, t, row) for s, t, row in candidates if s >= threshold]
    close.sort(reverse=True)

    return close[:k]

# 3. GENERATE STATS (like your theme_percentages.csv)
def get_theme_stats(df):
    themes = df['nss_labels_clean'].str.split(r'[,;]').explode().str.strip().str.lower()
    theme_counts = themes.value_counts()
    theme_pct = (theme_counts / len(df) * 100).round(2)
    theme_pct.to_csv("theme_percentages.csv")
    print("\n📊 theme_percentages.csv saved!")
    print(theme_pct.head(10))
    return theme_pct

theme_stats = get_theme_stats(df)

# 4. RAG PIPELINE FOR ANY THEME
def generate_report(theme, polarity="negative", k=3):
    """Full RAG: retrieve → select → report prompt"""
    query = f"{polarity} student feedback on {theme.lower()}"
    matching_df, embeddings, texts = retrieve(theme)

    inserts = select_inserts(query, matching_df, embeddings, texts, k=k)

    pct = (len(matching_df) / len(df)) * 100
    print(f"\n🎯 THEME: {theme}")
    print(f"📈 Stats: {len(matching_df)} mentions ({pct:.1f}%)")
    print(f"✅ Selected {len(inserts)} inserts (sim ≥0.65):")

    evidence = []
    for i, (sim, text, row) in enumerate(inserts):
        print(f"  {i+1}. Sim={sim:.3f} | {row['course_code']}: '{text[:100]}...'")
        evidence.append(f"Sim={sim:.2f}: {text}")

    # Report prompt (paste to your Llama)
    prompt = f"""Generate NSS-style report section for: {theme}
Polarity: {polarity.title()}
Stats: {len(matching_df)} mentions ({pct:.1f}% of {len(df)} reviews)

Use ONLY these {len(inserts)} evidence quotes:
{chr(10).join(evidence)}

Format like original tables: ranked subtopics, percentages, insights.
Keep under 100 words."""

    print("\n🤖 REPORT PROMPT (paste to Llama):\n" + prompt + "\n")
    return prompt, inserts

# =================================
# RUN FOR YOUR PAPER THEMES
# =================================
themes = [
    "Academic staff and support",
    "Covid-19 pandemic",
    "Learning opportunities",
    "Assessment and feedback"
]

for theme in themes:
    generate_report(theme, polarity="negative")

print("\n🎉 PIPELINE COMPLETE!")
print("1. theme_percentages.csv → update your tables")
print("2. REPORT PROMPTS → fine-tuned Llama → new RAG insights")
print("3. Show supervisor: 'Implemented insert selection via sim threshold'")


✅ Loaded 2000 reviews

Sample:
  course_code                                        review_text  \
0      CS 115                    go to office hours and practice   
1      CS 115  One of my least favourite courses. Although th...   
2      CS 115  It starts with a very low pace but after midte...   

                                    nss_labels_clean  
0  learning opportunities, assessment and feedbac...  
1  learning community, learning resources, studen...  
2                                learning community,  


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



📊 theme_percentages.csv saved!
nss_labels_clean
learning community             72.90
learning opportunities         41.85
academic staff and support     32.35
teaching                       32.30
                               30.30
assessment and feedback        27.90
organisation and management    27.55
covid-19 pandemic              20.85
overall                        17.35
learning resources             16.60
Name: count, dtype: float64

🎯 THEME: Academic staff and support
📈 Stats: 659 mentions (33.0%)
✅ Selected 0 inserts (sim ≥0.65):

🤖 REPORT PROMPT (paste to Llama):
Generate NSS-style report section for: Academic staff and support
Polarity: Negative
Stats: 659 mentions (33.0% of 2000 reviews)

Use ONLY these 0 evidence quotes:


Format like original tables: ranked subtopics, percentages, insights.
Keep under 100 words.


🎯 THEME: Covid-19 pandemic
📈 Stats: 420 mentions (21.0%)
✅ Selected 0 inserts (sim ≥0.65):

🤖 REPORT PROMPT (paste to Llama):
Generate NSS-style report secti

In [6]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity

# LOAD (your CSV already uploaded)
df = pd.read_csv("/content/uwaterloo_2000_classified.csv")
df['nss_labels_clean'] = df['nss_labels'].astype(str).str.lower().str.replace(r'labels?:\s*', '', regex=True)

model = SentenceTransformer('all-MiniLM-L6-v2')

def retrieve(theme, df=df):
    theme_lower = theme.lower()
    mask = df['nss_labels_clean'].str.contains(theme_lower, na=False)
    matching = df[mask].copy()
    if len(matching) == 0: return pd.DataFrame(), np.array([]), []
    texts = matching['review_text'].fillna('').astype(str).tolist()
    embeddings = model.encode(texts)
    return matching, embeddings, texts

def select_inserts(query, matching_df, embeddings, texts, k=3, threshold=0.55):  # LOWERED
    query_emb = model.encode(query)
    sims = cosine_similarity([query_emb], embeddings)[0]
    candidates = [(sims[i], texts[i], matching_df.iloc[i]) for i in range(len(texts))]
    close = [(s, t, row) for s, t, row in candidates if s >= threshold]
    close.sort(reverse=True)
    return close[:k]

# STATS
themes_flat = df['nss_labels_clean'].str.split(r'[,;]').explode().str.strip().str.lower()
theme_stats = (themes_flat.value_counts() / len(df) * 100).round(2)
theme_stats.to_csv("theme_percentages.csv")
print("📊 theme_percentages.csv:", theme_stats.head())

# FIXED RAG FUNCTION
def generate_report(theme, polarity="negative", k=3):
    # KEY FIX: theme-only query for selection (higher sim)
    selection_query = theme  # Just "Academic staff..." → perfect matches
    matching_df, embeddings, texts = retrieve(theme)

    inserts = select_inserts(selection_query, matching_df, embeddings, texts, k=k, threshold=0.55)

    pct = (len(matching_df) / len(df)) * 100
    print(f"\n🎯 {theme}")
    print(f"📈 {len(matching_df)} mentions ({pct:.1f}%)")
    print(f"✅ {len(inserts)} inserts (sim ≥0.55):")

    evidence = []
    for i, (sim, text, row) in enumerate(inserts):
        snippet = text[:100] + "..." if len(text) > 100 else text
        print(f"  {i+1}. Sim={sim:.3f} | {row['course_code']}: '{snippet}'")
        evidence.append(f"Sim={sim:.2f}: {text}")

    # Report prompt
    report_query = f"{polarity} feedback on {theme.lower()}"
    prompt = f"""NSS Report: {theme}
{polarity.title()} | {len(matching_df)} mentions ({pct:.1f}%)

Evidence ({len(inserts)} quotes):
{chr(10).join(evidence)}

Generate table-style insight (ranked subtopics, %s, <100 words)."""

    print("\n🤖 PROMPT:\n", prompt[:500], "...")
    return prompt

# RUN YOUR PAPER THEMES
themes = ["Academic staff and support", "Covid-19 pandemic", "Learning opportunities", "Assessment and feedback"]
for theme in themes:
    generate_report(theme, "negative")

print("\n🎉 FIXED! Now getting inserts for supervisor demo")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📊 theme_percentages.csv: nss_labels_clean
learning community            72.90
learning opportunities        41.85
academic staff and support    32.35
teaching                      32.30
                              30.30
Name: count, dtype: float64

🎯 Academic staff and support
📈 659 mentions (33.0%)
✅ 0 inserts (sim ≥0.55):

🤖 PROMPT:
 NSS Report: Academic staff and support
Negative | 659 mentions (33.0%)

Evidence (0 quotes):


Generate table-style insight (ranked subtopics, %s, <100 words). ...

🎯 Covid-19 pandemic
📈 420 mentions (21.0%)
✅ 0 inserts (sim ≥0.55):

🤖 PROMPT:
 NSS Report: Covid-19 pandemic
Negative | 420 mentions (21.0%)

Evidence (0 quotes):


Generate table-style insight (ranked subtopics, %s, <100 words). ...

🎯 Learning opportunities
📈 834 mentions (41.7%)
✅ 0 inserts (sim ≥0.55):

🤖 PROMPT:
 NSS Report: Learning opportunities
Negative | 834 mentions (41.7%)

Evidence (0 quotes):


Generate table-style insight (ranked subtopics, %s, <100 words). ...

🎯 Assessment 

In [7]:
# ========================================
# FINAL RAG - FUZZY MATCH + LOW THRESHOLD
# ========================================
!pip install sentence-transformers scikit-learn fuzzywuzzy python-levenshtein -q

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
from fuzzywuzzy import fuzz

# LOAD
df = pd.read_csv("/content/uwaterloo_2000_classified.csv")
df['nss_labels_clean'] = df['nss_labels'].astype(str).str.lower().str.replace(r'labels?:\s*', '', regex=True)

model = SentenceTransformer('all-MiniLM-L6-v2')

def fuzzy_retrieve(theme, df=df, min_score=70):
    """FUZZY MATCH themes (handles fragments like 'staff and suppo')"""
    theme_lower = theme.lower()
    mask = df['nss_labels_clean'].apply(lambda x: fuzz.partial_ratio(theme_lower, str(x)) >= min_score)
    matching = df[mask].copy()
    if len(matching) == 0:
        # Fallback: keyword contains
        mask = df['nss_labels_clean'].str.contains(theme_lower.replace(' and ', ' &?'), na=False, regex=True)
        matching = df[mask]
    if len(matching) == 0: return pd.DataFrame(), np.array([]), []
    texts = matching['review_text'].fillna('').astype(str).tolist()
    embeddings = model.encode(texts)
    return matching, embeddings, texts

def select_inserts(query, matching_df, embeddings, texts, k=3, threshold=0.45):  # LOWEST
    query_emb = model.encode(query)
    sims = cosine_similarity([query_emb], embeddings)[0]
    candidates = [(sims[i], texts[i], row) for i, row in enumerate(matching_df.itertuples())]
    close = [c for c in candidates if c[0] >= threshold]
    close.sort(reverse=True)
    return close[:k]

# STATS (cleaned)
themes_flat = df['nss_labels_clean'].str.split(r'[,;]').explode().str.strip().str.lower()
theme_stats = (themes_flat.value_counts() / len(df) * 100).round(2)
print("📊 theme_percentages.csv:", theme_stats.head(10))

# MAIN FUNCTION
def generate_report(theme, polarity="negative", k=3):
    selection_query = theme  # theme name only
    matching_df, embeddings, texts = fuzzy_retrieve(theme)

    inserts = select_inserts(selection_query, matching_df, embeddings, texts, k, 0.45)

    pct = len(matching_df) / len(df) * 100
    print(f"\n🎯 {theme} ({len(matching_df)}/{len(df)} = {pct:.1f}%)")
    print("✅ Top inserts:")

    evidence = []
    for i, (sim, text, row) in enumerate(inserts):
        snippet = (text[:80] + '...') if len(text) > 80 else text
        print(f"  {i+1}. Sim={sim:.3f} '{snippet}' [{row.Index}]")
        evidence.append(f"[{sim:.2f}] {text}")

    prompt = f"NSS {theme} | {polarity.title()} | {pct:.1f}%\n\n" + chr(10).join(evidence[:3])
    print("🤖 PROMPT:\n" + prompt)
    return inserts

# TEST ALL
themes = ["Academic staff and support", "Covid-19 pandemic", "Learning opportunities", "Assessment and feedback"]
for theme in themes:
    generate_report(theme)

print("\n🚀 NOW WORKING! Copy prompts → Llama")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 48.9 MB/s eta 0:00:00


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📊 theme_percentages.csv: nss_labels_clean
learning community             72.90
learning opportunities         41.85
academic staff and support     32.35
teaching                       32.30
                               30.30
assessment and feedback        27.90
organisation and management    27.55
covid-19 pandemic              20.85
overall                        17.35
learning resources             16.60
Name: count, dtype: float64

🎯 Academic staff and support (664/2000 = 33.2%)
✅ Top inserts:
🤖 PROMPT:
NSS Academic staff and support | Negative | 33.2%



🎯 Covid-19 pandemic (464/2000 = 23.2%)
✅ Top inserts:
  1. Sim=0.453 'Took it online during pandemic' [1658]
🤖 PROMPT:
NSS Covid-19 pandemic | Negative | 23.2%

[0.45] Took it online during pandemic

🎯 Learning opportunities (927/2000 = 46.4%)
✅ Top inserts:
  1. Sim=0.522 'intresting course' [1101]
  2. Sim=0.466 'Poorly taught, unengaging, and difficult at times (for a first year with no prio...' [358]
🤖 PROMPT:
NSS Learning op

In [8]:
def select_top_k(query, matching_df, embeddings, texts, k=3):
    """SIMPLE: Top-k by similarity (no threshold needed)"""
    query_emb = model.encode(query)
    sims = cosine_similarity([query_emb], embeddings)[0]
    idx = np.argsort(sims)[-k:][::-1]  # top k indices
    return [(sims[i], texts[i], matching_df.iloc[i]) for i in idx]

# Replace in generate_report:
inserts = select_top_k(theme, matching_df, embeddings, texts, k=3)


In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 58.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, util
import torch

# LOAD CSV → CHUNKS
df = pd.read_csv("/content/uwaterloo_2000_classified.csv")
rag_chunks = [
    f"Course: {row['course_code']} | Review: {row['review_text']} | Themes: {row['nss_labels']}"
    for _, row in df.iterrows()
]
print(f"✅ {len(rag_chunks)} Waterloo chunks")

# BUILD FAISS INDEX
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
embeddings = model.encode(rag_chunks, batch_size=64).astype('float32')
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

# BULLETPROOF SELECTION
def select_waterloo_inserts(query, top_k=8, final_k=3):
    query_emb = model.encode([query]).astype('float32')
    distances, indices = index.search(query_emb, top_k)

    # Safety checks
    if len(indices[0]) == 0 or indices[0][0] == -1:
        print("⚠️ No matches → fallback")
        return rag_chunks[:final_k]

    retrieved_embs = embeddings[indices[0]]
    sim_matrix = torch.nn.functional.cosine_similarity(
        torch.tensor(query_emb),
        torch.tensor(retrieved_embs),
        dim=1
    ).numpy()

    top_idx = np.argsort(sim_matrix)[-final_k:][::-1]
    selected = [rag_chunks[indices[0][i]] for i in top_idx]

    print(f"✅ Selected {len(selected)} | Best sim: {sim_matrix[top_idx[0]]:.3f}")
    for i, chunk in enumerate(selected):
        print(f"  {i+1}: {str(chunk)[:140]}")
    return selected

# TEST THEMES
themes = ["Academic staff and support", "Assessment and feedback", "Covid-19 pandemic"]
for theme in themes:
    print(f"\n🎯 {theme}")
    inserts = select_waterloo_inserts(f"{theme.lower()} student feedback")
    prompt = f"NSS {theme}:\n" + "\n\n".join(inserts)
    print("🤖 PROMPT:\n" + prompt[:400] + "...")  # ← fixed: proper f-string/concat

print("\n🎉 NO ERRORS - READY!")

✅ 2000 Waterloo chunks


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


🎯 Academic staff and support
✅ Selected 3 | Best sim: 0.772
  1: Course: STAT 230 | Review: Meh | Themes: Labels: Overall, Teaching, Academic staff and support, Organisation and management, Learning commun
  2: Course: ECON 101 | Review: Topics will appear in many other courses. | Themes: Labels: Assessment and feedback, Academic staff and support,
  3: Course: STAT 230 | Review: 5/7 would take again | Themes: Labels: Overall, Teaching, Academic staff and support, Organisation and management
🤖 PROMPT:
NSS Academic staff and support:
Course: STAT 230 | Review: Meh | Themes: Labels: Overall, Teaching, Academic staff and support, Organisation and management, Learning community, Student voice

Course: ECON 101 | Review: Topics will appear in many other courses. | Themes: Labels: Assessment and feedback, Academic staff and support,

Course: STAT 230 | Review: 5/7 would take again | Themes: Labels: O...

🎯 Assessment and feedback
✅ Selected 3 | Best sim: 0.800
  1: Course: ECON 101 | Review

In [4]:
"""
RAG Retrieval Pipeline - All NSS Themes
Outputs: nss_retrieved.json  (theme → top-3 chunks + metadata)
"""

import pandas as pd
import numpy as np
import faiss
import json
import re
from sentence_transformers import SentenceTransformer, util

# ── CONFIG ──────────────────────────────────────────────────────────────────
CSV_PATH   = "/content/uwaterloo_2000_classified.csv"
OUTPUT_JSON = "nss_retrieved.json"
TOP_K      = 10   # candidates from FAISS
FINAL_K    = 3    # kept per theme (re-ranked by cosine sim)

NSS_THEMES = [
    "The teaching on my course",
    "Learning opportunities",
    "Assessment and feedback",
    "Academic support",
    "Organisation and management",
    "Learning resources",
    "Student voice",
    "Student union",
    "Mental wellbeing",
    "Freedom of expression",
    "Academic staff and support",
    "Covid-19 pandemic",
]

# ── LOAD ────────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
rag_chunks = [
    f"Course: {row['course_code']} | Review: {row['review_text']} | Themes: {row['nss_labels']}"
    for _, row in df.iterrows()
]
print(f"✅ Loaded {len(rag_chunks)} chunks")

# ── EMBED ───────────────────────────────────────────────────────────────────
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
embeddings = model.encode(rag_chunks, batch_size=64).astype("float32")
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)
print("✅ FAISS index built")

# ── RETRIEVE ─────────────────────────────────────────────────────────────────
def retrieve(theme, top_k=TOP_K, final_k=FINAL_K):
    query = f"{theme.lower()} student feedback university experience"
    q_emb = model.encode([query]).astype("float32")
    D, I  = index.search(q_emb, top_k)

    if len(I[0]) == 0 or I[0][0] == -1:
        print(f"  ⚠️  No matches for '{theme}'")
        return rag_chunks[:final_k]

    ret_embs = embeddings[I[0]]
    sims = util.cos_sim(q_emb, ret_embs)[0].cpu().numpy()
    top_idx = np.argsort(sims)[-final_k:][::-1]

    chunks = []
    for idx in top_idx:
        raw = rag_chunks[I[0][idx]]
        # parse fields back out
        m = re.match(r"Course: (.+?) \| Review: (.+?) \| Themes: (.+)", raw)
        chunks.append({
            "course_code": m.group(1).strip() if m else "N/A",
            "review_text": m.group(2).strip() if m else raw,
            "nss_labels":  m.group(3).strip() if m else "",
            "similarity":  round(float(sims[idx]), 4),
        })

    best = chunks[0]["similarity"]
    print(f"  🎯 '{theme}' → best sim {best:.3f} | "
          f"top course: {chunks[0]['course_code']}")
    return chunks

# ── RUN ALL THEMES ───────────────────────────────────────────────────────────
results = {}
print("\n📋 Retrieving all NSS themes...\n")
for theme in NSS_THEMES:
    results[theme] = retrieve(theme)

# ── SAVE ────────────────────────────────────────────────────────────────────
with open(OUTPUT_JSON, "w") as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Saved → {OUTPUT_JSON}")
print("▶️  Next: node generate_report.js")

✅ Loaded 2000 chunks


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ FAISS index built

📋 Retrieving all NSS themes...

  🎯 'The teaching on my course' → best sim 0.789 | top course: STAT 230
  🎯 'Learning opportunities' → best sim 0.748 | top course: CS 246
  🎯 'Assessment and feedback' → best sim 0.766 | top course: ECON 101
  🎯 'Academic support' → best sim 0.745 | top course: CS 246
  🎯 'Organisation and management' → best sim 0.771 | top course: CS 246
  🎯 'Learning resources' → best sim 0.756 | top course: ECON 101
  🎯 'Student voice' → best sim 0.776 | top course: ECON 101
  🎯 'Student union' → best sim 0.720 | top course: CS 115
  🎯 'Mental wellbeing' → best sim 0.742 | top course: MATH 135
  🎯 'Freedom of expression' → best sim 0.716 | top course: CS 246
  🎯 'Academic staff and support' → best sim 0.755 | top course: STAT 230
  🎯 'Covid-19 pandemic' → best sim 0.845 | top course: CS 240

✅ Saved → nss_retrieved.json
▶️  Next: node generate_report.js


In [5]:
# Step 1 — in Colab, run rag_retrieve.py
# (copies your CSV path, runs retrieval for all 12 NSS themes, saves nss_retrieved.json)
exec(open("rag_retrieve.py").read())

# Step 2 — download nss_retrieved.json to the same folder as generate_report.js
# Then in terminal:
# node generate_report.js nss_retrieved.json

✅ Loaded 2000 chunks


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ FAISS index built

📋 Retrieving all NSS themes...

  🎯 'The teaching on my course' → best sim 0.789 | top course: STAT 230
  🎯 'Learning opportunities' → best sim 0.748 | top course: CS 246
  🎯 'Assessment and feedback' → best sim 0.766 | top course: ECON 101
  🎯 'Academic support' → best sim 0.745 | top course: CS 246
  🎯 'Organisation and management' → best sim 0.771 | top course: CS 246
  🎯 'Learning resources' → best sim 0.756 | top course: ECON 101
  🎯 'Student voice' → best sim 0.776 | top course: ECON 101
  🎯 'Student union' → best sim 0.720 | top course: CS 115
  🎯 'Mental wellbeing' → best sim 0.742 | top course: MATH 135
  🎯 'Freedom of expression' → best sim 0.716 | top course: CS 246
  🎯 'Academic staff and support' → best sim 0.755 | top course: STAT 230
  🎯 'Covid-19 pandemic' → best sim 0.845 | top course: CS 240

✅ Saved → nss_retrieved.json
▶️  Next: node generate_report.js


In [10]:
%%bash
cd /content
rm -rf node_modules package.json package-lock.json  # Clean previous
npm init -y
npm install docx fs path  # Core for .docx + JSON handling


Wrote to /content/package.json:

{
  "name": "content",
  "version": "1.0.0",
  "main": "generate_report.js",
  "scripts": {
    "test": "echo \"Error: no test specified\" && exit 1"
  },
  "keywords": [],
  "author": "",
  "license": "ISC",
  "description": ""
}




added 27 packages, and audited 28 packages in 4s

1 package is looking for funding
  run `npm fund` for details

found 0 vulnerabilities


In [11]:
!node generate_report.js nss_retrieved.json


✅ Loaded nss_retrieved.json

✅ Report written → NSS_Report.docx  (14.1 KB)
   Themes: 12 | Reviews/theme: 3
